# Model Evaluation

This notebook evaluates the invoice NER model by:
1. Calling the localhost API endpoint
2. Comparing predictions against ground truth labels from `test_labels.json`
3. Computing evaluation metrics (accuracy, precision, recall, F1)

In [112]:
import json
import requests
from pathlib import Path
from typing import Dict
import pandas as pd
from tqdm import tqdm

## Configuration

In [113]:
# API endpoint
API_URL = "http://localhost:7860"

# Data paths
TEST_JSON_PATH = "../data/test/test.json"
TEST_LABELS_PATH = "../data/SROIE2019/test/test_labels.json"
TEST_IMG_DIR = "../data/SROIE2019/test/img"
TEST_BOX_DIR = "../data/SROIE2019/test/box"

# TEST_JSON_PATH = "../data/SROIE2019/train/train.json"
# TEST_LABELS_PATH = "../data/SROIE2019/train/labels.json"
# TEST_IMG_DIR = "../data/SROIE2019/train/img"
# TEST_BOX_DIR = "../data/SROIE2019/train/box"

## Health Check

Verify the API is running

In [114]:
try:
    response = requests.get(f"{API_URL}/health", timeout=5)
    print(f"API Status: {response.json()}")
    if response.json().get('status') != 'healthy':
        print("⚠️ WARNING: API is not healthy!")
except Exception as e:
    print(f"❌ ERROR: Could not connect to API at {API_URL}")
    print(f"Error: {e}")
    print("\nMake sure the server is running with: uvicorn app:app --reload")

API Status: {'status': 'healthy', 'model_loaded': True, 'device': 'mps'}


## Load Test Data

In [115]:
# Load test dataset
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# Load ground truth labels
with open(TEST_LABELS_PATH, 'r', encoding='utf-8') as f:
    test_labels = json.load(f)

print(f"Loaded {len(test_data)} test samples")
print(f"Loaded {len(test_labels)} ground truth labels")
print(f"\nExample test data entry:")
print(f"  File: {test_data[0]['file']}")
print(f"  Words: {len(test_data[0]['words'])} words")
print(f"  Boxes: {len(test_data[0]['bboxes'])} boxes")
print(f"\nExample label:")
first_key = list(test_labels.keys())[0]
print(f"  {first_key}: {test_labels[first_key]}")

Loaded 347 test samples
Loaded 347 ground truth labels

Example test data entry:
  File: X51005675104.jpg
  Words: 169 words
  Boxes: 169 boxes

Example label:
  X00016469670.jpg: PEGIV-1030765


## Prediction Function

Function to call the API and get predictions

In [116]:
def predict(image_path: str, ocr_file_path: str) -> Dict:
    """
    Call the localhost API to get predictions using the new POST /predict endpoint
    
    Args:
        image_path: Path to the image file
        ocr_file_path: Path to the OCR text file (.txt format)
    
    Returns:
        Dictionary with prediction results
    """
    try:
        # Prepare files for upload
        with open(image_path, 'rb') as img_file, open(ocr_file_path, 'rb') as ocr_file:
            files = {
                'image': (Path(image_path).name, img_file, 'image/jpeg'),
                'ocr_file': (Path(ocr_file_path).name, ocr_file, 'text/plain')
            }
            
            # Call API
            response = requests.post(
                f"{API_URL}/predict",
                files=files,
                timeout=30
            )
        
        if response.status_code == 200:
            result = response.json()
            # Map API response to expected format
            return {
                "invoice_number": result.get("invoice_number"),
                "method": result.get("extraction_method"),
                "predictions": result.get("predictions", []),
                "total_words": result.get("total_words"),
                "model_device": result.get("model_device")
            }
        else:
            return {
                "error": f"API returned status {response.status_code}: {response.text}",
                "invoice_number": None
            }
    
    except Exception as e:
        return {
            "error": str(e),
            "invoice_number": None
        }

## Run Evaluation

Predict on all test samples and compare with ground truth

In [117]:
# Run predictions
results = []
errors = []

for item in tqdm(test_data, desc="Running predictions"):
    filename = item['file']
    image_path = Path(TEST_IMG_DIR) / filename
    
    # Construct OCR text file path (assuming same name with .txt extension)
    ocr_file_path = Path(str(image_path).replace('/img/', '/box/')).with_suffix('.txt')
    
    # Skip if no ground truth label or label is "ambiguous". Log it
    if filename not in test_labels:
        errors.append({
            'file': filename,
            'error': 'No ground truth label'
        })
        continue

    if test_labels[filename] == "ambiguous":
        errors.append({
            'file': filename,
            'error': 'Ambiguous label - skipped'
        })
        continue
    
    # Skip if OCR file doesn't exist
    if not ocr_file_path.exists():
        errors.append({
            'file': filename,
            'error': f'OCR file not found: {ocr_file_path}'
        })
        continue
    
    # Get ground truth
    ground_truth = test_labels[filename]
    
    # Get prediction
    prediction_result = predict(
        str(image_path),
        str(ocr_file_path)
    )
    
    predicted = prediction_result.get('invoice_number')
    
    # Check for errors
    if 'error' in prediction_result:
        errors.append({
            'file': filename,
            'error': prediction_result['error']
        })
        print(prediction_result['error'])
        break
    
    # Store result
    results.append({
        'file': filename,
        'ground_truth': ground_truth,
        'predicted': predicted,
        'method': prediction_result.get('method', 'unknown'),
        'total_words': prediction_result.get('total_words', 0),
        'match': ground_truth == predicted if predicted else False
    })

print(f"\n✅ Completed {len(results)} predictions")
if errors:
    print(f"⚠️ {len(errors)} errors occurred")

Running predictions: 100%|██████████| 347/347 [00:42<00:00,  8.17it/s]


✅ Completed 323 predictions
⚠️ 24 errors occurred


## Compute Metrics

In [118]:
# Convert to DataFrame for analysis
df = pd.DataFrame(results)

# Add ambiguous flag
df['is_ambiguous'] = df['ground_truth'] == 'ambiguous'

# Add empty prediction flag
df['is_empty'] = df['predicted'].apply(lambda x: x in [None, '', 'Not Found'] or pd.isna(x))

# Add human review flag (predictions with >1 word need review)
df['needs_human_review'] = df['predicted'].apply(lambda x: len(str(x).split()) > 1 if pd.notna(x) and x not in [None, '', 'Not Found'] else False)

# Overall accuracy (excluding ambiguous and empty predictions)
df_valid = df[~df['is_ambiguous'] & ~df['is_empty']]
total = len(df_valid)
correct = df_valid['match'].sum()
accuracy = correct / total if total > 0 else 0

# Accuracy without human review cases
df_no_review = df_valid[~df_valid['needs_human_review']]
total_no_review = len(df_no_review)
correct_no_review = df_no_review['match'].sum()
accuracy_no_review = correct_no_review / total_no_review if total_no_review > 0 else 0

# Count predictions by method (valid only)
method_counts = df_valid['method'].value_counts()

# Accuracy by method (excluding empty predictions)
accuracy_by_method = {}
for method in df_valid['method'].unique():
    method_df = df_valid[df_valid['method'] == method]
    method_correct = method_df['match'].sum()
    method_total = len(method_df)
    method_accuracy = method_correct / method_total if method_total > 0 else 0
    accuracy_by_method[method] = {
        'correct': method_correct,
        'total': method_total,
        'accuracy': method_accuracy
    }

# Count None/empty predictions
df_non_ambiguous = df[~df['is_ambiguous']]
none_predictions = df_non_ambiguous['predicted'].isna().sum()
empty_predictions = df_non_ambiguous['predicted'].apply(lambda x: x in [None, '', 'Not Found'] if pd.notna(x) else False).sum()
no_result_total = none_predictions + empty_predictions

# Count human review cases
human_review_count = df_valid['needs_human_review'].sum()

# Separate skipped samples from actual errors
skipped_no_label = sum(1 for err in errors if err['error'] == 'No ground truth label')
skipped_ambiguous = sum(1 for err in errors if err['error'] == 'Ambiguous label - skipped')
actual_errors = [err for err in errors if err['error'] not in ['No ground truth label', 'Ambiguous label - skipped']]

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"\nTotal samples in dataset: {len(df)}")
print(f"  - Ambiguous labels: {df['is_ambiguous'].sum()}")
print(f"  - Empty/No predictions: {no_result_total}")
print(f"  - Valid for evaluation: {total}")
print(f"Skipped samples: {skipped_no_label}")
print(f"  - No ground truth label: {skipped_no_label}")
print(f"\nCorrect predictions: {correct}/{total}")
print(f"Accuracy (valid predictions only): {accuracy:.2%}")
print(f"\n👁️ Human review needed (>1 word): {human_review_count} ({human_review_count/total:.1%})")
print(f"Accuracy (excluding human review): {correct_no_review}/{total_no_review} = {accuracy_no_review:.2%}")
print(f"\nPredictions by method:")
for method, count in method_counts.items():
    print(f"  {method}: {count} ({count/total:.1%})")
print(f"\nAccuracy by method:")
for method, stats in accuracy_by_method.items():
    print(f"  {method}: {stats['correct']}/{stats['total']} = {stats['accuracy']:.2%}")
print(f"\nEmpty/No predictions (excluded from accuracy): {no_result_total} ({no_result_total/len(df_non_ambiguous):.1%})")
print(f"  - None: {none_predictions}")
print(f"  - Empty/Not Found: {empty_predictions}")

if actual_errors:
    print(f"\n❌ Actual errors: {len(actual_errors)}")
    print("\nFirst 10 errors:")
    for err in actual_errors[:10]:
        print(f"  {err['file']}: {err['error']}")

EVALUATION RESULTS

Total samples in dataset: 323
  - Ambiguous labels: 0
  - Empty/No predictions: 7
  - Valid for evaluation: 316
Skipped samples: 0
  - No ground truth label: 0

Correct predictions: 294/316
Accuracy (valid predictions only): 93.04%

👁️ Human review needed (>1 word): 20 (6.3%)
Accuracy (excluding human review): 289/296 = 97.64%

Predictions by method:
  heuristic: 191 (60.4%)
  model: 125 (39.6%)

Accuracy by method:
  model: 106/125 = 84.80%
  heuristic: 188/191 = 98.43%

Empty/No predictions (excluded from accuracy): 7 (2.2%)
  - None: 0
  - Empty/Not Found: 7


## Analyze Errors

Look at mismatches to understand failure modes

In [119]:
# Get mismatches (excluding ambiguous and empty)
mismatches = df[~df['match'] & ~df['is_ambiguous'] & ~df['is_empty']].copy()

print(f"\nTotal mismatches (non-ambiguous, non-empty): {len(mismatches)}\n")
print("First 20 mismatches:")
print("=" * 80)

for idx, row in mismatches.head(20).iterrows():
    print(f"File: {row['file']}")
    print(f"  Ground Truth: {row['ground_truth']}")
    print(f"  Predicted:    {row['predicted']}")
    print(f"  Method:       {row['method']}")
    print()


Total mismatches (non-ambiguous, non-empty): 22

First 20 mismatches:
File: X51005568885.jpg
  Ground Truth: 139110
  Predicted:    139110 01 -
  Method:       model

File: X51006828200.jpg
  Ground Truth: 001-1402935
  Predicted:    1000758 001-1402935
  Method:       model

File: X51005568892.jpg
  Ground Truth: 142507
  Predicted:    142507 -
  Method:       model

File: X51006334766.jpg
  Ground Truth: 00014603 / 10P01
  Predicted:    00014603
  Method:       heuristic

File: X51007846387.jpg
  Ground Truth: MR-T01105105
  Predicted:    MR-T01105105-T01105105 9555047308127
  Method:       model

File: X51006401977.jpg
  Ground Truth: 00133886 / POS01
  Predicted:    00133886
  Method:       heuristic

File: X51005719889.jpg
  Ground Truth: TB011530
  Predicted:    TB011530 180104074210_TB011530_DJF8537854+23
  Method:       model

File: X51006008082.jpg
  Ground Truth: CS00534185
  Predicted:    CS00534185 CS00530344
  Method:       model

File: X51005749904.jpg
  Ground Truth: 01

## Categorize Errors

In [120]:
def categorize_error(row):
    """Categorize the type of error"""
    gt = str(row['ground_truth']) if row['ground_truth'] else ""
    pred = str(row['predicted']) if row['predicted'] else ""
    
    if not pred or pred == 'None':
        return 'No prediction'
    elif gt in pred or pred in gt:
        return 'Partial match'
    elif gt.replace(' ', '') == pred.replace(' ', ''):
        return 'Spacing difference'
    elif gt.replace('-', '') == pred.replace('-', ''):
        return 'Delimiter difference'
    else:
        return 'Complete mismatch'

mismatches['error_category'] = mismatches.apply(categorize_error, axis=1)

print("\nError categories:")
print(mismatches['error_category'].value_counts())
print()

# Show examples of each category
for category in mismatches['error_category'].unique():
    print(f"\n{category} examples:")
    examples = mismatches[mismatches['error_category'] == category].head(3)
    for _, row in examples.iterrows():
        print(f"  {row['file']}: '{row['ground_truth']}' vs '{row['predicted']}'")


Error categories:
error_category
Partial match        21
Complete mismatch     1
Name: count, dtype: int64


Partial match examples:
  X51005568885.jpg: '139110' vs '139110 01 -'
  X51006828200.jpg: '001-1402935' vs '1000758 001-1402935'
  X51005568892.jpg: '142507' vs '142507 -'

Complete mismatch examples:
  X51005433556.jpg: '003-1220845' vs '1000249-1220845'


In [121]:
complete_mismatches = mismatches[mismatches['error_category'] == 'Complete mismatch'].copy()
print(f"\n\n{'='*80}")
print(f"COMPLETE MISMATCHES: {len(complete_mismatches)}")
print(f"{'='*80}\n")

# Display the dataframe
complete_mismatches[['file', 'ground_truth', 'predicted', 'method', 'needs_human_review']]



COMPLETE MISMATCHES: 1



,file,ground_truth,predicted,method,needs_human_review
200,X51005433556.jpg,003-1220845,1000249-1220845,model,False


In [123]:
# Get mismatches (excluding ambiguous, empty, and human review cases)
mismatches = df[~df['match'] & ~df['is_ambiguous'] & ~df['is_empty'] & ~df['needs_human_review']].copy()

print(f"\nTotal mismatches (non-ambiguous, non-empty, no human review): {len(mismatches)}\n")
print("=" * 80)

mismatches[['file', 'ground_truth', 'predicted', 'method', 'needs_human_review']]


Total mismatches (non-ambiguous, non-empty, no human review): 7



,file,ground_truth,predicted,method,needs_human_review
43,X51006334766.jpg,00014603 / 10P01,00014603,heuristic,False
58,X51006401977.jpg,00133886 / POS01,00133886,heuristic,False
187,X51005719855.jpg,063975,A063975,model,False
191,X51006327953.jpg,00015258 / 10P01,00015258,heuristic,False
200,X51005433556.jpg,003-1220845,1000249-1220845,model,False
274,X51007579725.jpg,001-1112563,001-1112563-1112563,model,False
310,X51006008206.jpg,001-731709,001-731709-731709,model,False
